In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.formula.api as smf
import statsmodels.stats.descriptivestats as smd
import statsmodels.api as sm
import scipy.stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sys
sys.path.append('../')
import plotting

colormap = {
    "no": "#969696",
    "3m": "#de2d26",
    "5m": "#fcae91",
}

# Load qPCR data

In [ ]:
dfs = []
for exp in ("Exp1", "Exp2", "Exp3"):
    df = pd.read_csv(f"../data/qpcr/Primer3_{exp}.csv")
    df['exp'] = exp
    dfs.append(df)

df = pd.concat(dfs)
df

# Remove NTCs and dilutions above a Ct of 30

In [ ]:
df = df[df['sample'] != "ntc"].copy()

df = df[df['Ct'] < 30].copy()

df

# Assess dilution-by-dilution Delta-Ct

In [ ]:
# for each experiment and sample, assess the difference in ct between dilutions
plot_df = df.copy()

# per sample, exp, and dilution, calculate the mean Ct
plot_df = plot_df.groupby(['exp', 'sample', 'dilution']).agg(Ct=('Ct', 'mean')).reset_index()

# calculate the difference in Ct between dilutions
plot_df['delta_Ct'] = plot_df.groupby(['exp', 'sample'])['Ct'].diff()

# calculate the mean and standard deviation of delta_Ct
plot_df['mean_delta_Ct'] = plot_df.groupby(['dilution', 'sample'])['delta_Ct'].transform('mean')
plot_df['std_delta_Ct'] = plot_df.groupby(['dilution', 'sample'])['delta_Ct'].transform('std')

fig = px.scatter(
    plot_df, 
    x="dilution", 
    y="mean_delta_Ct", 
    error_y="std_delta_Ct",
    color="sample",
    trendline="ols",
    color_discrete_map=colormap,
)

fig.update_yaxes(
    title="Δ(Cycle threshold) between dilutions",
    range=[3, 5],
)

fig.update_layout(
    margin=dict(l=0, r=0, t=12, b=0),
    height=300,
    width=680,
    showlegend=False,
)
fig.update_xaxes(title="log10(dilution)", range=[1.9, 7.1])
fig = plotting.standardize_plot(fig)
fig.show()
fig.write_image("./SI_figure_primer3_sequences/qpcr_deltact.svg")

# Plot calibration curves

In [ ]:
fig = px.scatter(
    df, 
    x="dilution", 
    y="Ct", 
    color="sample", 
    facet_col="exp",
    trendline="ols",
    color_discrete_map=colormap,
)
fig.update_layout(
    yaxis_title="Cycle threshold",
    margin=dict(l=0, r=0, t=12, b=0),
    height=300,
    width=680,
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(
    title="log10(dilution)",
)

fig = plotting.standardize_plot(fig)
fig.write_image("./SI_figure_primer3_sequences/qpcr_ct.svg")
fig.show()

# also save the data
df.to_csv("./SI_figure_primer3_sequences/qpcr_ct.csv", index=False)

# Find regression data

In [ ]:
data = px.get_trendline_results(fig).copy()
data[["slope", "intercept", "R2"]] = data.px_fit_results.apply(lambda x: pd.Series({"slope": x.params[1], "intercept": x.params[0], "R2": x.rsquared}))
data['eff_exp'] = 10**(1/data['slope'])-1

data

In [ ]:
res = {}
for group in data['sample'].unique():
    idata = data[data['sample'] == group]
    res[group] = smd.describe(idata['eff_exp'], stats=['mean', 'ci'], use_t=True)['eff_exp']
    res[group]['delta_ci'] = res[group]['upper_ci'] - res[group]['mean']
pd.DataFrame(res.values(), index=res.keys())

# Plot experimental efficiency

In [ ]:
result = data.groupby(['sample'], as_index=False).agg({'eff_exp':['mean','std']})
result.columns = ['sample', 'mean', 'std']

cut_interval = [0.05, 0.55]
bar = px.bar(
    result,
    x="sample",
    y="mean",
    color_discrete_map=colormap,
)
fig = make_subplots(
    rows=2,
    cols=1,
    row_heights=[0.8, 0.2],
    vertical_spacing=0.05,
    shared_xaxes=True,
)

fig.add_traces(bar.data, rows=[1]*len(bar.data), cols=[1]*len(bar.data))
fig.add_traces(bar.data, rows=[2]*len(bar.data), cols=[1]*len(bar.data))

fig.update_yaxes(range=[cut_interval[1], 1], row=1, col=1)
fig.update_xaxes(visible=False, row=1, col=1)
fig.update_yaxes(range=[0, cut_interval[0]], row=2, col=1)

fig.add_trace(
    px.scatter(
        data,
        x="sample",
        y="eff_exp",
        color_discrete_sequence=["black"]
    ).data[0]
)


fig.update_layout(
    xaxis_title="Sequence ID",
    yaxis_title="qPCR efficiency",
    width=200,
    height=200,
    margin=dict(l=0, r=10, t=60, b=0),
    showlegend=False,
)

fig.update_yaxes(
    tickformat=',.0%', 
    dtick=0.2,
    minor_dtick=0.1,
)

fig = plotting.standardize_plot(fig)
fig.write_image("SI_figure_primer3_sequences/qpcr_efficiency.svg")
fig.show()

In [ ]:
result

# One-way ANOVA + Tukeys range test

In [ ]:
scipy.stats.levene(
    *[data.loc[data['sample'] == sid, "eff_exp"].values for sid in data['sample'].unique()], 
    center='median'
)

In [ ]:
m = smf.ols('eff_exp ~ C(sample)', data=data).fit()
display(m.summary())

display(sm.stats.anova_lm(m, typ=2))

In [ ]:
posthoc = pairwise_tukeyhsd(endog = data["eff_exp"], groups = data["sample"])
display(posthoc.summary())
display(posthoc.pvalues)